# 53 — Chunking Strategies

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/53-chunking-strategies/chunking_workbook.ipynb)

```
Document
   |
   +-- Fixed-size    -> uniform blocks
   +-- Recursive     -> respects structure
   +-- Sentence-win  -> sentence + neighbors
   +-- Semantic      -> embedding boundaries
        |
     Vectorstore -> Retrieve -> Generate
```

## What you will learn

- How four chunking strategies split the same document differently
- Why chunk boundaries matter for retrieval precision
- How to pick the right strategy based on your content type
- How to measure the practical impact of chunking on RAG answer quality

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain-text-splitters==0.3.11",
        "langchain-openai==0.3.33",
        "langchain-chroma==0.2.6",
        "python-dotenv==1.1.1",
    ], check=True)

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

In [ ]:
SAMPLE_TEXT = """Neural networks are computing systems loosely inspired by biological neural networks. Deep learning uses multiple layers to progressively extract higher-level features from raw input.

The transformer architecture introduced in 2017 revolutionized natural language processing. It replaces recurrent networks with self-attention, allowing all tokens to attend to each other simultaneously. This parallelism enables training on much larger datasets.

Retrieval-augmented generation (RAG) combines a language model with an external knowledge base. When a question arrives, a retriever fetches relevant documents, and the generator conditions its output on both the question and the retrieved context. This grounds the model in factual sources.

Vector databases store high-dimensional embeddings and support approximate nearest neighbor search. Systems like ChromaDB, Qdrant, and Pinecone make it practical to search millions of document chunks in milliseconds. The embedding quality is as important as the retrieval algorithm.

Evaluation of RAG pipelines requires specialized metrics. Faithfulness checks whether the answer is grounded in the retrieved context. Context recall checks whether all necessary information was retrieved. Answer relevancy checks whether the answer directly addresses the question."""

print(f"Document: {len(SAMPLE_TEXT)} characters, {len(SAMPLE_TEXT.split())} words")
print(f"Paragraphs: {len([p for p in SAMPLE_TEXT.split(chr(10)*2) if p.strip()])}")

---

## Part 1 — Fixed-Size Chunking

Split by character count with a fixed `chunk_size` and `chunk_overlap`.

**Why Not Just This?** It ignores sentence and paragraph boundaries. A chunk might end mid-sentence, cutting context exactly where the answer lives.

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

fixed_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator="")
fixed_chunks = fixed_splitter.split_text(SAMPLE_TEXT)

print(f"Fixed-size chunks: {len(fixed_chunks)}")
for i, c in enumerate(fixed_chunks):
    print(f"  [{i}] {len(c)} chars | {repr(c[:60])}...")

---

## Part 2 — Recursive Chunking

Tries separators in order: `\n\n` -> `\n` -> ` ` -> `""`. Splits on the largest boundary that fits, falling back to smaller ones only when needed.

**Why Not Just This?** Best general-purpose choice for prose. Preserves paragraph and sentence structure. Fails for code or tables where structure is different.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
recursive_chunks = recursive_splitter.split_text(SAMPLE_TEXT)

print(f"Recursive chunks: {len(recursive_chunks)}")
for i, c in enumerate(recursive_chunks):
    print(f"  [{i}] {len(c)} chars | {repr(c[:60])}...")

---

## Part 3 — Sentence-Window Chunking

Split into individual sentences, then embed each sentence but store it with its surrounding `window_size` neighbors as context. Small chunk = precise retrieval; large window = rich generation context.

**Why Not Just This?** Adds overhead: each sentence gets extra context stored with it. Index size grows with window size. Pairs well with parent document retriever.

In [ ]:
import re

def sentence_window_chunks(text: str, window: int = 1) -> list[dict]:
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
    result = []
    for i, sent in enumerate(sentences):
        lo, hi = max(0, i - window), min(len(sentences), i + window + 1)
        result.append({"sentence": sent, "context": " ".join(sentences[lo:hi])})
    return result

sw_chunks = sentence_window_chunks(SAMPLE_TEXT, window=1)
print(f"Sentence-window chunks: {len(sw_chunks)}")
for item in sw_chunks[:3]:
    print(f"  Embed: {repr(item['sentence'][:50])}")
    print(f"  Store: {repr(item['context'][:80])}")
    print()

---

## Part 4 — Semantic Chunking (Embedding-Based)

Embed each sentence; compute cosine similarity between consecutive sentences. Where similarity drops sharply, there is a topic shift — split there.

**Why Not Just This?** Requires an embedding call per sentence — slower and costs tokens. But produces chunks aligned to actual topic shifts, not arbitrary character counts.

In [ ]:
import re
import numpy as np
from langchain_openai import OpenAIEmbeddings

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))

def semantic_chunks(text: str, threshold: float = 0.75) -> list[str]:
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
    emb_model = OpenAIEmbeddings(model="text-embedding-3-small")
    embeddings = emb_model.embed_documents(sentences)
    chunks, current = [], [sentences[0]]
    for i in range(1, len(sentences)):
        sim = cosine(embeddings[i - 1], embeddings[i])
        if sim < threshold:
            chunks.append(" ".join(current))
            current = [sentences[i]]
        else:
            current.append(sentences[i])
    if current:
        chunks.append(" ".join(current))
    return chunks

print("Running semantic chunking (makes embedding API calls)...")
semantic = semantic_chunks(SAMPLE_TEXT, threshold=0.75)
print(f"Semantic chunks: {len(semantic)}")
for i, c in enumerate(semantic):
    print(f"  [{i}] {len(c)} chars | {repr(c[:70])}...")

---

## Comparison Table

| Strategy | Respects structure | Needs embeddings | Chunk count | Best for |
|----------|-------------------|-----------------|-------------|----------|
| Fixed-size | No | No | Most | Simple baseline, uniform index size |
| Recursive | Yes | No | Fewer | General prose, markdown, code |
| Sentence-window | Yes | No | Most (per sentence) | Precise recall + rich context |
| Semantic | Yes | Yes | Fewest | Multi-topic docs, presentations |

In [ ]:
sw_text_chunks = [item["sentence"] for item in sw_chunks]

rows = [
    ("Fixed-size",       fixed_chunks,     "No",  "No"),
    ("Recursive",        recursive_chunks, "Yes", "No"),
    ("Sentence-window",  sw_text_chunks,   "Yes", "No"),
    ("Semantic",         semantic,         "Yes", "Yes"),
]

print(f"{'Strategy':<20} {'Chunks':>6} {'Avg chars':>9} {'Structure':>10} {'Embeddings':>11}")
print("-" * 62)
for name, chunks, structure, needs_emb in rows:
    avg = sum(len(c) for c in chunks) // len(chunks) if chunks else 0
    print(f"{name:<20} {len(chunks):>6} {avg:>9} {structure:>10} {needs_emb:>11}")